# QLoRA для темы «Дом и квартира»

В Google Colab на T4 выполнены короткая проверка и два полных обучения на 80 диалогах. В обоих экспериментах получено ухудшение относительно базы. Исходные результаты сохранены в results/sft_20261104T183742Z_9bc87f.

Для воспроизведения откройте новую сессию с T4 через меню «Среда выполнения → Сменить среду выполнения». Загрузите архив artifacts/home-sft-colab.zip в следующей ячейке: он содержит зафиксированные данные v1 и скрипты.

Выполняйте ячейки сверху вниз. Сначала проверьте данные и короткое обучение на восьми примерах. Затем включите переключатели полного обучения и validation. Второй эксперимент отличается только learning rate. Все запуски создают новые папки результатов.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Загрузите один архив home-sft-colab.zip")
filename, payload = next(iter(uploaded.items()))
PROJECT = Path("/content/home-safety-sft")
if PROJECT.exists():
    raise FileExistsError("Папка проекта уже существует. Используйте текущие файлы или новую сессию.")
with ZipFile(BytesIO(payload)) as archive:
    for member in archive.infolist():
        destination = (PROJECT / member.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError("Недопустимый путь в архиве")
    PROJECT.mkdir()
    archive.extractall(PROJECT)
os.chdir(PROJECT)
print(PROJECT)


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run(["nvidia-smi"], check=True)


## Проверка данных и длины

Каждый запуск создаёт отдельную папку. В полном обучении loss вычисляется по ответам ассистента. Системный текст, вопросы и дополнение до длины батча маскируются значением `-100`.


In [ ]:
from datetime import datetime, timezone
import uuid
import json

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN = PROJECT / "results" / ("sft_" + RUN_ID)
RUN.mkdir(parents=True, exist_ok=False)

def run_script(name, *args):
    subprocess.run([sys.executable, "src/" + name, *map(str, args)], check=True)

run_script("validate_data.py", "--data-dir", "data/v1", "--output-dir", RUN / "data_check", "--load-with-datasets")
run_script("prepare_sft.py", "--data-dir", "data/v1", "--output-dir", RUN / "tokenization")
print(json.loads((RUN / "tokenization/summary.json").read_text())["splits"])


## Короткий запуск

Дождитесь статуса `completed`, двух шагов оптимизатора и `reload_verified=true`. В папке сохраняются конечная loss, логи, память GPU, конфигурация и ответ после загрузки адаптера с диска. Loss короткого запуска служит проверкой работоспособности. Качество модели оценивается позже на отдельных вопросах.


In [ ]:
run_script("train_qlora.py", "--mode", "sanity", "--data-dir", "data/v1", "--output-dir", RUN / "sanity")
print((RUN / "sanity/run.json").read_text())
print((RUN / "sanity/reload_probe.json").read_text())


## Полное обучение

Начальная конфигурация использует 80 примеров, 2 эпохи, rank 16, alpha 32, learning rate 0.0001, batch size 1 и накопление градиента 8. Это примерно 20 шагов оптимизатора. Короткий и полный запуски каждый раз загружают исходные веса Qwen. Подбор последующих настроек опирается на validation.

После проверки данных и короткого запуска установите `RUN_FULL_TRAINING = True` и выполните ячейку. Сохраните артефакты до отключения среды Colab.


In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    sanity = json.loads((RUN / "sanity/run.json").read_text())
    if sanity.get("status") != "completed" or not sanity.get("reload_verified"):
        raise RuntimeError("Сначала завершите короткий запуск и проверку загрузки адаптера")
    run_script("train_qlora.py", "--mode", "train", "--data-dir", "data/v1", "--output-dir", RUN / "full")
    print((RUN / "full/run.json").read_text())
else:
    print("Полное обучение ожидает переключения RUN_FULL_TRAINING.")


## Сравнение на validation

После полного обучения установите `RUN_VALIDATION = True`. Обе модели отвечают на 40 одинаковых вопросов с закреплёнными параметрами генерации. Ответы сохраняются по одному, вместе с токенами и признаками достижения лимита. Оценка моделью-судьёй проводится отдельно. Автоматический критерий применим к 2 из 40 вопросов, поэтому рядом с долей прошедших проверку указывайте знаменатель.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    full_status = json.loads((RUN / "full/run.json").read_text())
    assert full_status["status"] == "completed" and full_status["reload_verified"]
    run_script("evaluate_model.py", "--split", "validation", "--output-dir", RUN / "validation_base")
    run_script("evaluate_model.py", "--split", "validation", "--adapter", RUN / "full/adapter", "--output-dir", RUN / "validation_sft")
    for variant in ("base", "sft"):
        run_script("checks.py", "--answers", RUN / f"validation_{variant}/answers.jsonl",
                   "--annotations", "data/v1/validation_annotations.jsonl",
                   "--examples", "data/safety_check_examples.jsonl",
                   "--output-dir", RUN / f"validation_{variant}/automatic_check")
    run_script("summarize_validation.py", "--base", RUN / "validation_base", "--sft", RUN / "validation_sft",
               "--annotations", "data/v1/validation_annotations.jsonl", "--output-dir", RUN / "validation_summary")
    print("Ответы validation и автоматическая проверка сохранены.")


## Повторный эксперимент с меньшим learning rate

После ухудшения на validation проверяется learning rate 0.00002. Остальные настройки, данные, вопросы и генерация сохраняются. Для воспроизведения включите `RUN_LOWER_LR_EXPERIMENT = True`. Базовые ответы используются повторно.

In [ ]:
RUN_LOWER_LR_EXPERIMENT = False
if RUN_LOWER_LR_EXPERIMENT:
    assert json.loads((RUN / "validation_base/run.json").read_text())["status"] == "completed"
    assert json.loads((RUN / "validation_sft/run.json").read_text())["status"] == "completed"
    run_script("train_qlora.py", "--mode", "train", "--learning-rate", "0.00002",
               "--data-dir", "data/v1", "--output-dir", RUN / "full_lr2e5")
    run_script("evaluate_model.py", "--split", "validation", "--adapter", RUN / "full_lr2e5/adapter",
               "--output-dir", RUN / "validation_sft_lr2e5")
    run_script("checks.py", "--answers", RUN / "validation_sft_lr2e5/answers.jsonl",
               "--annotations", "data/v1/validation_annotations.jsonl",
               "--examples", "data/safety_check_examples.jsonl",
               "--output-dir", RUN / "validation_sft_lr2e5/automatic_check")
    run_script("summarize_validation.py", "--base", RUN / "validation_base", "--sft", RUN / "validation_sft_lr2e5",
               "--annotations", "data/v1/validation_annotations.jsonl", "--output-dir", RUN / "validation_summary_lr2e5")


## Скачать результаты

Эту ячейку можно запускать и после ошибки. Архив содержит доступные логи, traceback и сохранённые веса. Полный адаптер находится в `full/adapter/adapter_model.safetensors`, короткий в `sanity/adapter/`. Итоговое сравнение требует отдельного запуска исходной модели и адаптера на одинаковых вопросах и с одинаковыми параметрами генерации.


In [ ]:
import shutil
archive_path = shutil.make_archive(str(RUN), "zip", root_dir=RUN.parent, base_dir=RUN.name)
print(archive_path)
files.download(archive_path)
